# Question 1 (A)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import sys
import os
import time

# Try to import matplotlib, but don't fail if missing (fallback to just logging)
try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False

# ------------------------------------------------------------------
# 0. Logging Setup
# ------------------------------------------------------------------
class Logger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w", buffering=1) # 'w' to overwrite, 'a' to append

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# Set up logging to file
log_file = "experiment_log.txt"
if not isinstance(sys.stdout, Logger):
    sys.stdout = Logger(log_file)

print(f"Starting experiment at {time.ctime()}")
print(f"Logs will be saved to {os.path.abspath(log_file)}")


# ------------------------------------------------------------------
# Global tracker for best accuracy across all runs
GLOBAL_BEST_ACC = 0.0

# ------------------------------------------------------------------
# 1. Dataset Setup (70-10-20 Split)
# ------------------------------------------------------------------
def prepare_data(dataset_name, batch_size, pin_mem):
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    data_root = '/csehome/m25csa013/data'

    if dataset_name == "MNIST":
        full_ds = datasets.MNIST(root=data_root, train=True, download=True, transform=transform)
        test_ds = datasets.MNIST(root=data_root, train=False, download=True, transform=transform)
    else:
        full_ds = datasets.FashionMNIST(root=data_root, train=True, download=True, transform=transform)
        test_ds = datasets.FashionMNIST(root=data_root, train=False, download=True, transform=transform)

    combined = torch.utils.data.ConcatDataset([full_ds, test_ds])

    # 70-10-20 split of 70,000 images
    train_size = 49000
    val_size = 7000
    test_size = 14000

    if len(combined) != (train_size + val_size + test_size):
        # Fallback if dataset sizes differ
        total = len(combined)
        val_size = int(0.10 * total)
        test_size = int(0.20 * total)
        train_size = total - val_size - test_size

    train_set, val_set, test_set = random_split(combined, [train_size, val_size, test_size])

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              pin_memory=pin_mem, num_workers=4)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False,
                            pin_memory=pin_mem, num_workers=4)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False,
                             pin_memory=pin_mem, num_workers=4)

    return train_loader, val_loader, test_loader

# ------------------------------------------------------------------
# 2. Utils: Plotting and Training
# ------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def save_plots(history, filename_prefix):
    if not HAS_MATPLOTLIB:
        return

    epochs = range(1, len(history['train_loss']) + 1)

    # Loss Plot
    plt.figure(figsize=(10, 5))
    plt.plot(epochs, history['train_loss'], label='Train Loss')
    plt.plot(epochs, history['val_loss'], label='Val Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    os.makedirs("plots", exist_ok=True)
    plt.savefig(f"plots/{filename_prefix}_loss.png")
    plt.close()

    # Accuracy Plot
    plt.figure(figsize=(10, 5))
    plt.plot(epochs, history['train_acc'], label='Train Acc')
    plt.plot(epochs, history['val_acc'], label='Val Acc')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True)
    plt.savefig(f"plots/{filename_prefix}_acc.png")
    plt.close()

def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, (correct / total) * 100

def train_config(model_type, dataset_name, bs, opt_name, lr, epoch_limit, pin_mem):
    # Model Init
    if model_type == "ResNet-18":
        model = models.resnet18(num_classes=10)
    else:
        model = models.resnet50(num_classes=10)
    model = model.to(device)

    # Optimizer
    if opt_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    criterion = nn.CrossEntropyLoss()

    # Scaler
    scaler = None
    if device.type == 'cuda':
        try:
            # New standard
            scaler = torch.amp.GradScaler('cuda')
        except AttributeError:
            # Fallback for older torch versions
            scaler = torch.cuda.amp.GradScaler()

    train_loader, val_loader, test_loader = prepare_data(dataset_name, bs, pin_mem)

    # Tracking
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0.0

    # Identifier for file naming
    config_id = f"{dataset_name}_{model_type}_{opt_name}_lr{lr}_bs{bs}"

    # Training Loop
    for epoch in range(1, epoch_limit + 1):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            if device.type == 'cuda' and scaler:
                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

            # Batch metrics
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

        # Epoch Metrics
        train_loss = running_loss / total
        train_acc = (correct / total) * 100

        # Validation
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        # Save Best Model Logic (Global Best Only)
        global GLOBAL_BEST_ACC
        if val_acc > best_val_acc: # Updates local best (for this run)
            best_val_acc = val_acc

        # Check against global best to save space
        if val_acc > GLOBAL_BEST_ACC:
            GLOBAL_BEST_ACC = val_acc
            # Ensure folder exists
            os.makedirs("models", exist_ok=True)
            torch.save(model.state_dict(), "models/best_model_q1a.pth")

    # Plotting
    save_plots(history, config_id)

    # Final Test
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    return test_acc, best_val_acc

# ------------------------------------------------------------------
# Main Execution
# ------------------------------------------------------------------
if __name__ == "__main__":
    print(f"✅ Implementation using device: {device}")

    # Configuration Lists
    datasets_to_test = ["MNIST", "FashionMNIST"]
    models_to_test = ["ResNet-18", "ResNet-50"]

    table_configs = [
        (16, "SGD", 0.001),
        (16, "SGD", 0.0001),
        (16, "Adam", 0.001),
        (16, "Adam", 0.0001),
        (32, "SGD", 0.001),
        (32, "Adam", 0.0001)
    ]

    epoch_choices = [3, 5]
    pin_memory_choices = [True, False]

    print("\n" + "="*140)
    print(f"{'Dataset':<12} | {'BS':<3} | {'Opt':<5} | {'LR':<7} | {'PinMem':<6} | {'Epochs':<6} | {'Model':<10} | {'Test Acc':<8} | {'Best Val':<8}")
    print("="*140)

    for d_name in datasets_to_test:
        for pin_mem in pin_memory_choices:
            for epochs in epoch_choices:
                print(f"--- Group: {d_name} | PinMem: {pin_mem} | Epochs: {epochs} ---")

                for (bs, opt, lr) in table_configs:
                    for model_name in models_to_test:
                        try:
                            # Start timer
                            t0 = time.time()

                            test_acc, best_val = train_config(
                                model_type=model_name,
                                dataset_name=d_name,
                                bs=bs,
                                opt_name=opt,
                                lr=lr,
                                epoch_limit=epochs,
                                pin_mem=pin_mem
                            )

                            elapsed = time.time() - t0
                            print(f"{d_name:<12} | {bs:<3} | {opt:<5} | {lr:<7} | {pin_mem!s:<6} | {epochs:<6} | {model_name:<10} | {test_acc:.2f}%     | {best_val:.2f}% (Time: {elapsed:.1f}s)")

                        except Exception as e:
                            print(f"ERROR: {model_name} {bs} {opt} {lr} - {e}")
                            import traceback
                            traceback.print_exc()

    print("\nExperiments Completed.")

```text
Starting experiment at Thu Jan 22 01:49:41 2026
Logs will be saved to /csehome/m25csa013/MLOps/experiment_log.txt
✅ Implementation using device: cuda

============================================================================================================================================
Dataset      | BS  | Opt   | LR      | PinMem | Epochs | Model      | Test Acc | Best Val
============================================================================================================================================
--- Group: MNIST | PinMem: True | Epochs: 3 ---
MNIST        | 16  | SGD   | 0.001   | True   | 3      | ResNet-18  | 99.02%     | 99.10% (Time: 143.1s)
MNIST        | 16  | SGD   | 0.001   | True   | 3      | ResNet-50  | 98.48%     | 98.40% (Time: 226.9s)
MNIST        | 16  | SGD   | 0.0001  | True   | 3      | ResNet-18  | 98.31%     | 98.40% (Time: 142.5s)
MNIST        | 16  | SGD   | 0.0001  | True   | 3      | ResNet-50  | 97.06%     | 96.77% (Time: 227.4s)
MNIST        | 16  | Adam  | 0.001   | True   | 3      | ResNet-18  | 99.11%     | 99.14% (Time: 147.8s)
MNIST        | 16  | Adam  | 0.001   | True   | 3      | ResNet-50  | 97.91%     | 98.47% (Time: 242.4s)
MNIST        | 16  | Adam  | 0.0001  | True   | 3      | ResNet-18  | 98.87%     | 98.84% (Time: 147.5s)
MNIST        | 16  | Adam  | 0.0001  | True   | 3      | ResNet-50  | 98.34%     | 98.51% (Time: 243.2s)
MNIST        | 32  | SGD   | 0.001   | True   | 3      | ResNet-18  | 98.94%     | 99.09% (Time: 104.7s)
MNIST        | 32  | SGD   | 0.001   | True   | 3      | ResNet-50  | 98.71%     | 98.60% (Time: 146.8s)
MNIST        | 32  | Adam  | 0.0001  | True   | 3      | ResNet-18  | 98.60%     | 98.67% (Time: 109.2s)
MNIST        | 32  | Adam  | 0.0001  | True   | 3      | ResNet-50  | 98.24%     | 98.24% (Time: 154.2s)
--- Group: MNIST | PinMem: True | Epochs: 5 ---
MNIST        | 16  | SGD   | 0.001   | True   | 5      | ResNet-18  | 99.33%     | 99.30% (Time: 235.0s)
MNIST        | 16  | SGD   | 0.001   | True   | 5      | ResNet-50  | 98.78%     | 98.81% (Time: 372.7s)
MNIST        | 16  | SGD   | 0.0001  | True   | 5      | ResNet-18  | 98.47%     | 98.56% (Time: 234.2s)
MNIST        | 16  | SGD   | 0.0001  | True   | 5      | ResNet-50  | 97.98%     | 98.34% (Time: 374.0s)
MNIST        | 16  | Adam  | 0.001   | True   | 5      | ResNet-18  | 99.36%     | 99.31% (Time: 242.9s)
MNIST        | 16  | Adam  | 0.001   | True   | 5      | ResNet-50  | 98.68%     | 98.99% (Time: 399.3s)
MNIST        | 16  | Adam  | 0.0001  | True   | 5      | ResNet-18  | 99.32%     | 99.30% (Time: 241.7s)
MNIST        | 16  | Adam  | 0.0001  | True   | 5      | ResNet-50  | 98.76%     | 98.87% (Time: 399.0s)
MNIST        | 32  | SGD   | 0.001   | True   | 5      | ResNet-18  | 99.16%     | 99.06% (Time: 169.5s)
MNIST        | 32  | SGD   | 0.001   | True   | 5      | ResNet-50  | 98.92%     | 98.86% (Time: 238.8s)
MNIST        | 32  | Adam  | 0.0001  | True   | 5      | ResNet-18  | 98.99%     | 99.07% (Time: 177.5s)
MNIST        | 32  | Adam  | 0.0001  | True   | 5      | ResNet-50  | 98.59%     | 98.67% (Time: 252.6s)
--- Group: MNIST | PinMem: False | Epochs: 3 ---
MNIST        | 16  | SGD   | 0.001   | False  | 3      | ResNet-18  | 99.10%     | 98.69% (Time: 148.2s)
MNIST        | 16  | SGD   | 0.001   | False  | 3      | ResNet-50  | 98.73%     | 98.59% (Time: 229.1s)
MNIST        | 16  | SGD   | 0.0001  | False  | 3      | ResNet-18  | 98.36%     | 98.29% (Time: 143.8s)
MNIST        | 16  | SGD   | 0.0001  | False  | 3      | ResNet-50  | 97.26%     | 97.34% (Time: 228.3s)
MNIST        | 16  | Adam  | 0.001   | False  | 3      | ResNet-18  | 99.04%     | 99.00% (Time: 149.2s)
MNIST        | 16  | Adam  | 0.001   | False  | 3      | ResNet-50  | 98.59%     | 98.61% (Time: 243.8s)
MNIST        | 16  | Adam  | 0.0001  | False  | 3      | ResNet-18  | 98.70%     | 99.01% (Time: 149.3s)
MNIST        | 16  | Adam  | 0.0001  | False  | 3      | ResNet-50  | 98.01%     | 98.14% (Time: 243.9s)
MNIST        | 32  | SGD   | 0.001   | False  | 3      | ResNet-18  | 98.96%     | 99.10% (Time: 104.9s)
MNIST        | 32  | SGD   | 0.001   | False  | 3      | ResNet-50  | 98.66%     | 98.53% (Time: 151.9s)
MNIST        | 32  | Adam  | 0.0001  | False  | 3      | ResNet-18  | 98.79%     | 98.79% (Time: 108.5s)
MNIST        | 32  | Adam  | 0.0001  | False   | 3      | ResNet-50  | 98.60%     | 98.33% (Time: 155.0s)
--- Group: MNIST | PinMem: False | Epochs: 5 ---
MNIST        | 16  | SGD   | 0.001   | False  | 5      | ResNet-18  | 99.09%     | 99.11% (Time: 235.9s)
MNIST        | 16  | SGD   | 0.001   | False  | 5      | ResNet-50  | 99.05%     | 99.07% (Time: 375.0s)
MNIST        | 16  | SGD   | 0.0001  | False  | 5      | ResNet-18  | 98.61%     | 98.80% (Time: 234.5s)
MNIST        | 16  | SGD   | 0.0001  | False  | 5      | ResNet-50  | 98.21%     | 98.37% (Time: 374.9s)
MNIST        | 16  | Adam  | 0.001   | False  | 5      | ResNet-18  | 99.16%     | 99.09% (Time: 243.5s)
MNIST        | 16  | Adam  | 0.001   | False  | 5      | ResNet-50  | 98.80%     | 98.81% (Time: 398.9s)
MNIST        | 16  | Adam  | 0.0001  | False  | 5      | ResNet-18  | 99.15%     | 99.20% (Time: 243.6s)
MNIST        | 16  | Adam  | 0.0001  | False  | 5      | ResNet-50  | 98.74%     | 98.80% (Time: 400.9s)
MNIST        | 32  | SGD   | 0.001   | False  | 5      | ResNet-18  | 99.06%     | 99.07% (Time: 170.8s)
MNIST        | 32  | SGD   | 0.001   | False  | 5      | ResNet-50  | 98.72%     | 98.89% (Time: 240.3s)
MNIST        | 32  | Adam  | 0.0001  | False  | 5      | ResNet-18  | 98.89%     | 99.01% (Time: 175.9s)
MNIST        | 32  | Adam  | 0.0001  | False  | 5      | ResNet-50  | 98.69%     | 98.89% (Time: 251.8s)
--- Group: FashionMNIST | PinMem: True | Epochs: 3 ---
FashionMNIST | 16  | SGD   | 0.001   | True   | 3      | ResNet-18  | 90.18%     | 90.44% (Time: 144.0s)
FashionMNIST | 16  | SGD   | 0.001   | True   | 3      | ResNet-50  | 89.49%     | 89.11% (Time: 227.8s)
FashionMNIST | 16  | SGD   | 0.0001  | True   | 3      | ResNet-18  | 89.05%     | 89.03% (Time: 143.4s)
FashionMNIST | 16  | SGD   | 0.0001  | True   | 3      | ResNet-50  | 85.04%     | 84.53% (Time: 229.5s)
FashionMNIST | 16  | Adam  | 0.001   | True   | 3      | ResNet-18  | 90.93%     | 90.67% (Time: 148.8s)
FashionMNIST | 16  | Adam  | 0.001   | True   | 3      | ResNet-50  | 88.55%     | 88.64% (Time: 243.6s)
FashionMNIST | 16  | Adam  | 0.0001  | True   | 3      | ResNet-18  | 90.45%     | 90.83% (Time: 148.5s)
FashionMNIST | 16  | Adam  | 0.0001  | True   | 3      | ResNet-50  | 87.74%     | 88.69% (Time: 243.1s)
FashionMNIST | 32  | SGD   | 0.001   | True   | 3      | ResNet-18  | 90.64%     | 89.97% (Time: 105.3s)
FashionMNIST | 32  | SGD   | 0.001   | True   | 3      | ResNet-50  | 88.55%     | 87.87% (Time: 146.6s)
FashionMNIST | 32  | Adam  | 0.0001  | True   | 3      | ResNet-18  | 91.54%     | 91.26% (Time: 109.0s)
FashionMNIST | 32  | Adam  | 0.0001  | True   | 3      | ResNet-50  | 88.39%     | 88.79% (Time: 154.2s)
--- Group: FashionMNIST | PinMem: True | Epochs: 5 ---
FashionMNIST | 16  | SGD   | 0.001   | True   | 5      | ResNet-18  | 91.56%     | 91.97% (Time: 233.1s)
FashionMNIST | 16  | SGD   | 0.001   | True   | 5      | ResNet-50  | 90.55%     | 90.74% (Time: 373.9s)
FashionMNIST | 16  | SGD   | 0.0001  | True   | 5      | ResNet-18  | 89.96%     | 90.16% (Time: 234.2s)
FashionMNIST | 16  | SGD   | 0.0001  | True   | 5      | ResNet-50  | 87.08%     | 86.89% (Time: 373.7s)
FashionMNIST | 16  | Adam  | 0.001   | True   | 5      | ResNet-18  | 91.49%     | 92.03% (Time: 241.6s)
FashionMNIST | 16  | Adam  | 0.001   | True   | 5      | ResNet-50  | 91.09%     | 91.13% (Time: 398.1s)
FashionMNIST | 16  | Adam  | 0.0001  | True   | 5      | ResNet-18  | 91.56%     | 91.73% (Time: 241.7s)
FashionMNIST | 16  | Adam  | 0.0001  | True   | 5      | ResNet-50  | 91.31%     | 91.50% (Time: 399.0s)
FashionMNIST | 32  | SGD   | 0.001   | True   | 5      | ResNet-18  | 91.49%     | 91.12% (Time: 170.5s)
FashionMNIST | 32  | SGD   | 0.001   | True   | 5      | ResNet-50  | 89.28%     | 89.44% (Time: 240.2s)
FashionMNIST | 32  | Adam  | 0.0001  | True   | 5      | ResNet-18  | 92.05%     | 91.88% (Time: 177.1s)
FashionMNIST | 32  | Adam  | 0.0001  | True   | 5      | ResNet-50  | 89.50%     | 89.76% (Time: 253.9s)
--- Group: FashionMNIST | PinMem: False | Epochs: 3 ---
FashionMNIST | 16  | SGD   | 0.001   | False  | 3      | ResNet-18  | 90.15%     | 90.38% (Time: 144.5s)
FashionMNIST | 16  | SGD   | 0.001   | False  | 3      | ResNet-50  | 89.35%     | 89.05% (Time: 228.1s)
FashionMNIST | 16  | SGD   | 0.0001  | False  | 3      | ResNet-18  | 89.12%     | 89.08% (Time: 144.1s)
FashionMNIST | 16  | SGD   | 0.0001  | False  | 3      | ResNet-50  | 85.11%     | 84.67% (Time: 230.2s)
FashionMNIST | 16  | Adam  | 0.001   | False  | 3      | ResNet-18  | 91.01%     | 90.72% (Time: 149.3s)
FashionMNIST | 16  | Adam  | 0.001   | False  | 3      | ResNet-50  | 88.62%     | 88.58% (Time: 244.0s)
FashionMNIST | 16  | Adam  | 0.0001  | False  | 3      | ResNet-18  | 90.55%     | 90.79% (Time: 149.0s)
FashionMNIST | 16  | Adam  | 0.0001  | False  | 3      | ResNet-50  | 87.89%     | 88.75% (Time: 243.8s)
FashionMNIST | 32  | SGD   | 0.001   | False  | 3      | ResNet-18  | 90.66%     | 90.02% (Time: 106.1s)
FashionMNIST | 32  | SGD   | 0.001   | False  | 3      | ResNet-50  | 88.60%     | 87.94% (Time: 147.2s)
FashionMNIST | 32  | Adam  | 0.0001  | False  | 3      | ResNet-18  | 91.49%     | 91.31% (Time: 109.8s)
FashionMNIST | 32  | Adam  | 0.0001  | False  | 3      | ResNet-50  | 88.45%     | 88.85% (Time: 154.9s)
--- Group: FashionMNIST | PinMem: False | Epochs: 5 ---
FashionMNIST | 16  | SGD   | 0.001   | False  | 5      | ResNet-18  | 91.60%     | 91.95% (Time: 234.5s)
FashionMNIST | 16  | SGD   | 0.001   | False  | 5      | ResNet-50  | 90.62%     | 90.81% (Time: 375.1s)
FashionMNIST | 16  | SGD   | 0.0001  | False  | 5      | ResNet-18  | 90.05%     | 90.22% (Time: 235.1s)
FashionMNIST | 16  | SGD   | 0.0001  | False  | 5      | ResNet-50  | 87.15%     | 87.01% (Time: 374.8s)
FashionMNIST | 16  | Adam  | 0.001   | False  | 5      | ResNet-18  | 91.55%     | 92.10% (Time: 242.8s)
FashionMNIST | 16  | Adam  | 0.001   | False  | 5      | ResNet-50  | 91.15%     | 91.20% (Time: 399.5s)
FashionMNIST | 16  | Adam  | 0.0001  | False  | 5      | ResNet-18  | 91.61%     | 91.79% (Time: 242.9s)
FashionMNIST | 16  | Adam  | 0.0001  | False  | 5      | ResNet-50  | 91.38%     | 91.56% (Time: 400.2s)
FashionMNIST | 32  | SGD   | 0.001   | False  | 5      | ResNet-18  | 91.45%     | 91.08% (Time: 172.1s)
FashionMNIST | 32  | SGD   | 0.001   | False  | 5      | ResNet-50  | 89.34%     | 89.51% (Time: 241.6s)
FashionMNIST | 32  | Adam  | 0.0001  | False  | 5      | ResNet-18  | 91.95%     | 91.75% (Time: 178.6s)
FashionMNIST | 32  | Adam  | 0.0001  | False  | 5      | ResNet-50  | 89.62%     | 89.81% (Time: 254.3s)

Experiments Completed.
```

# Question 1 (B)

In [ ]:
import torch
from torchvision import datasets, transforms
import time
import sys
import os
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import joblib

GLOBAL_BEST_ACC = 0.0

# ------------------------------------------------------------------
# 0. Logging Setup
# ------------------------------------------------------------------
class Logger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w", buffering=1)

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

log_file = "svm_log.txt"
if not isinstance(sys.stdout, Logger):
    sys.stdout = Logger(log_file)

# ------------------------------------------------------------------
# 1. Data Preparation (Flattened)
# ------------------------------------------------------------------
def get_data(dataset_name):
    transform = transforms.Compose([
        transforms.Resize((28, 28)), # Ensure 28x28
        transforms.ToTensor(), # [0, 1]
        transforms.Lambda(lambda x: x.view(-1)) # Flatten to 784 vector
    ])

    data_root = '/csehome/m25csa013/data'

    if dataset_name == "MNIST":
        train_ds = datasets.MNIST(root=data_root, train=True, download=True, transform=transform)
        test_ds = datasets.MNIST(root=data_root, train=False, download=True, transform=transform)
    else:
        train_ds = datasets.FashionMNIST(root=data_root, train=True, download=True, transform=transform)
        test_ds = datasets.FashionMNIST(root=data_root, train=False, download=True, transform=transform)

    print(f"Loading {dataset_name} (Full Dataset)...")

    # Use full dataset
    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=len(train_ds), shuffle=False)
    test_loader = torch.utils.data.DataLoader(test_ds, batch_size=len(test_ds), shuffle=False)

    X_train, y_train = next(iter(train_loader))
    X_test, y_test = next(iter(test_loader))

    X_train, y_train = X_train.numpy(), y_train.numpy()
    X_test, y_test = X_test.numpy(), y_test.numpy()

    print(f"Data Loaded: Train {X_train.shape}, Test {X_test.shape}")
    return X_train, y_train, X_test, y_test

# ------------------------------------------------------------------
# 2. Training Function
# ------------------------------------------------------------------
def train_svm(dataset_name, kernel, X_train, y_train, X_test, y_test):
    print(f"\n>>> Training SVM on {dataset_name} with Kernel: {kernel} (CPU)")

    t0 = time.time()

    # Sklearn SVC
    model = SVC(kernel=kernel, C=1.0)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    acc = accuracy_score(y_test, predictions) * 100

    t1 = time.time()
    train_time_ms = (t1 - t0) * 1000

    print(f"Result: Acc={acc:.2f}%, Time={train_time_ms:.2f}ms")

    # Save Model (Global Best Only)
    global GLOBAL_BEST_ACC
    if acc > GLOBAL_BEST_ACC:
        GLOBAL_BEST_ACC = acc
        os.makedirs("models", exist_ok=True)
        save_path = "models/best_model_q1b.pkl"
        joblib.dump(model, save_path)

    return acc, train_time_ms

# ------------------------------------------------------------------
# Main
# ------------------------------------------------------------------
if __name__ == "__main__":
    print(f"Starting SVM Experiments (CPU Mode)...")

    datasets_list = ["MNIST", "FashionMNIST"]
    kernels = ["poly", "rbf"]

    print("\n" + "="*80)
    print(f"{'Dataset':<15} | {'Kernel':<10} | {'Test Acc (%)':<15} | {'Time (ms)':<15}")
    print("="*80)

    for d_name in datasets_list:
        X_train, y_train, X_test, y_test = get_data(d_name)

        for k in kernels:
            try:
                acc, t_ms = train_svm(d_name, k, X_train, y_train, X_test, y_test)
                print(f"{d_name:<15} | {k:<10} | {acc:<15.2f} | {t_ms:<15.2f}")
            except Exception as e:
                print(f"ERROR: {d_name} {k} - {e}")

```text
Starting SVM Experiments (CPU Mode)...

================================================================================
Dataset         | Kernel     | Test Acc (%)    | Time (ms)      
================================================================================
Loading MNIST (Full Dataset)...
Data Loaded: Train (60000, 784), Test (10000, 784)

>>> Training SVM on MNIST with Kernel: poly (CPU)
Result: Acc=97.71%, Time=233846.10ms
MNIST           | poly       | 97.71           | 233846.10      

>>> Training SVM on MNIST with Kernel: rbf (CPU)
Result: Acc=97.92%, Time=272000.61ms
MNIST           | rbf        | 97.92           | 272000.61      
Loading FashionMNIST (Full Dataset)...
Data Loaded: Train (60000, 784), Test (10000, 784)

>>> Training SVM on FashionMNIST with Kernel: poly (CPU)
Result: Acc=86.30%, Time=393137.24ms
FashionMNIST    | poly       | 86.30           | 393137.24      

>>> Training SVM on FashionMNIST with Kernel: rbf (CPU)
Result: Acc=88.29%, Time=397992.41ms
FashionMNIST    | rbf        | 88.29           | 397992.41
```

# Question 2

## CPU

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import time
import sys
import os

GLOBAL_BEST_ACC = 0.0

# ------------------------------------------------------------------
# 0. Logging & Setup
# ------------------------------------------------------------------
class Logger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w", buffering=1)

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

log_file = "benchmark_log_cpu.txt"
if not isinstance(sys.stdout, Logger):
    sys.stdout = Logger(log_file)

# ------------------------------------------------------------------
# 1. FLOPs Counter (Custom)
# ------------------------------------------------------------------
def count_flops(model, input_size=(1, 3, 64, 64), device='cpu'):
    flops = 0

    def register_hooks(module):
        def hook(mod, inp, out):
            nonlocal flops
            if isinstance(mod, nn.Conv2d):
                in_c = mod.in_channels
                out_c = mod.out_channels
                k_h, k_w = mod.kernel_size
                b, c, h, w = out.shape
                f = 2 * in_c * k_h * k_w * out_c * h * w
                if mod.bias is not None:
                     f += out_c * h * w
                flops += (f // b)
            elif isinstance(mod, nn.Linear):
                in_f = mod.in_features
                out_f = mod.out_features
                f = 2 * in_f * out_f
                if mod.bias is not None:
                    f += out_f
                flops += f

        if not isinstance(module, nn.Sequential) and \
           not isinstance(module, models.resnet.ResNet) and \
           not isinstance(module, models.resnet.BasicBlock) and \
           not isinstance(module, models.resnet.Bottleneck):
              return module.register_forward_hook(hook)
        return None

    handler_list = []
    for m in model.modules():
        h = register_hooks(m)
        if h: handler_list.append(h)

    dummy_input = torch.randn(input_size).to(device)
    model.to(device)
    model.eval()
    with torch.no_grad():
        model(dummy_input)

    for h in handler_list:
        h.remove()

    return flops

# ------------------------------------------------------------------
# 2. Benchmarking Function
# ------------------------------------------------------------------
def run_benchmark(model_name, device_str, bs, opt_name, lr):
    print(f"\n>>> Benchmark: {model_name} on {device_str.upper()} | BS={bs} | Opt={opt_name} | LR={lr}")

    device = torch.device(device_str)

    # 1. Model init
    if model_name == "ResNet-18":
        model = models.resnet18(num_classes=10)
    elif model_name == "ResNet-50":
        model = models.resnet50(num_classes=10)
    else:
        print("Note: ResNet-32 not standard in torchvision, using ResNet-18 as proxy/placeholder or skipping.")
        model = models.resnet18(num_classes=10)

    model = model.to(device)

    # 2. Calculate FLOPs
    flops = count_flops(model, input_size=(1, 3, 64, 64), device=device)
    flops_g = flops / 1e9
    print(f"   FLOPs (per image): {flops_g:.4f} GFLOPs")

    # 3. Data (FashionMNIST as per Q2)
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    data_root = '/csehome/m25csa013/data'
    ds_train = datasets.FashionMNIST(root=data_root, train=True, download=True, transform=transform)
    ds_test = datasets.FashionMNIST(root=data_root, train=False, download=True, transform=transform)

    # Use Full Dataset
    train_loader = DataLoader(ds_train, batch_size=bs, shuffle=True, num_workers=4, pin_memory=False)
    test_loader = DataLoader(ds_test, batch_size=bs, shuffle=False, num_workers=4, pin_memory=False)

    # 4. Training (1 Epoch Loop)
    if opt_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    criterion = nn.CrossEntropyLoss()

    model.train()

    t0 = time.time()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    t1 = time.time()
    train_time_ms = (t1 - t0) * 1000


    # 5. Test Accuracy
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    accuracy = (correct / total) * 100
    print(f"   Test Accuracy: {accuracy:.2f}%")

    # Save Model (Global Best Only)
    global GLOBAL_BEST_ACC
    if accuracy > GLOBAL_BEST_ACC:
        GLOBAL_BEST_ACC = accuracy
        os.makedirs("models", exist_ok=True)
        save_path = "models/best_model_q2_cpu.pth"
        torch.save(model.state_dict(), save_path)

    return train_time_ms, flops_g, accuracy

# ------------------------------------------------------------------
# Main
# ------------------------------------------------------------------
if __name__ == "__main__":

    configs = [
        # CPU
        ('cpu', 16, 'SGD', 0.001),
        ('cpu', 16, 'Adam', 0.001),
    ]

    models_to_test = ["ResNet-18", "ResNet-50"]

    print("\n" + "="*120)
    print(f"{'Device':<6} | {'Model':<10} | {'BS':<3} | {'Opt':<5} | {'LR':<7} | {'Train Time (ms)':<16} | {'GFLOPs':<10} | {'Acc (%)':<8}")
    print("="*120)

    for device_str, bs, opt, lr in configs:
        if device_str == 'cuda' and not torch.cuda.is_available():
            continue

        for model_name in models_to_test:
            try:
                t_ms, gflops, acc = run_benchmark(model_name, device_str, bs, opt, lr)
                print(f"{device_str:<6} | {model_name:<10} | {bs:<3} | {opt:<5} | {lr:<7} | {t_ms:<16.2f} | {gflops:<10.4f} | {acc:<8.2f}")
            except Exception as e:
                print(f"ERROR: {model_name} on {device_str} - {e}")
                # import traceback
                # traceback.print_exc()

```text
========================================================================================================================
Device | Model      | BS  | Opt   | LR      | Train Time (ms)  | GFLOPs     | Acc (%)
========================================================================================================================

>>> Benchmark: ResNet-18 on CPU | BS=16 | Opt=SGD | LR=0.001
   FLOPs (per image): 0.2961 GFLOPs
   Test Accuracy: 87.65%
cpu    | ResNet-18  | 16  | SGD   | 0.001   | 135015.12        | 0.2961     | 87.65   

>>> Benchmark: ResNet-50 on CPU | BS=16 | Opt=SGD | LR=0.001
   FLOPs (per image): 0.6673 GFLOPs
   Test Accuracy: 85.63%
cpu    | ResNet-50  | 16  | SGD   | 0.001   | 346542.64        | 0.6673     | 85.63   

>>> Benchmark: ResNet-18 on CPU | BS=16 | Opt=Adam | LR=0.001
   FLOPs (per image): 0.2961 GFLOPs
   Test Accuracy: 88.19%
cpu    | ResNet-18  | 16  | Adam  | 0.001   | 165550.63        | 0.2961     | 88.19   

>>> Benchmark: ResNet-50 on CPU | BS=16 | Opt=Adam | LR=0.001
   FLOPs (per image): 0.6673 GFLOPs
   Test Accuracy: 87.13%
cpu    | ResNet-50  | 16  | Adam  | 0.001   | 420020.88        | 0.6673     | 87.13
```

## GPU

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import time
import sys
import os

GLOBAL_BEST_ACC = 0.0

# ------------------------------------------------------------------
# 0. Logging & Setup
# ------------------------------------------------------------------
class Logger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w", buffering=1)

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

log_file = "benchmark_log_gpu.txt"
if not isinstance(sys.stdout, Logger):
    sys.stdout = Logger(log_file)

# ------------------------------------------------------------------
# 1. FLOPs Counter (Custom)
# ------------------------------------------------------------------
def count_flops(model, input_size=(1, 3, 64, 64), device='cpu'):
    flops = 0

    def register_hooks(module):
        def hook(mod, inp, out):
            nonlocal flops
            if isinstance(mod, nn.Conv2d):
                in_c = mod.in_channels
                out_c = mod.out_channels
                k_h, k_w = mod.kernel_size
                b, c, h, w = out.shape
                f = 2 * in_c * k_h * k_w * out_c * h * w
                if mod.bias is not None:
                     f += out_c * h * w
                flops += (f // b)
            elif isinstance(mod, nn.Linear):
                in_f = mod.in_features
                out_f = mod.out_features
                f = 2 * in_f * out_f
                if mod.bias is not None:
                    f += out_f
                flops += f

        if not isinstance(module, nn.Sequential) and \
           not isinstance(module, models.resnet.ResNet) and \
           not isinstance(module, models.resnet.BasicBlock) and \
           not isinstance(module, models.resnet.Bottleneck):
              return module.register_forward_hook(hook)
        return None

    handler_list = []
    for m in model.modules():
        h = register_hooks(m)
        if h: handler_list.append(h)

    dummy_input = torch.randn(input_size).to(device)
    model.to(device)
    model.eval()
    with torch.no_grad():
        model(dummy_input)

    for h in handler_list:
        h.remove()

    return flops

# ------------------------------------------------------------------
# 2. Benchmarking Function
# ------------------------------------------------------------------
def run_benchmark(model_name, device_str, bs, opt_name, lr):
    print(f"\n>>> Benchmark: {model_name} on {device_str.upper()} | BS={bs} | Opt={opt_name} | LR={lr}")

    device = torch.device(device_str)
    if device_str == 'cuda':
        torch.backends.cudnn.benchmark = True

    # 1. Model init
    if model_name == "ResNet-18":
        model = models.resnet18(num_classes=10)
    elif model_name == "ResNet-50":
        model = models.resnet50(num_classes=10)
    else:
        print("Note: ResNet-32 not standard in torchvision, using ResNet-18 as proxy/placeholder or skipping.")
        model = models.resnet18(num_classes=10)

    model = model.to(device)

    # 2. Calculate FLOPs
    flops = count_flops(model, input_size=(1, 3, 64, 64), device=device)
    flops_g = flops / 1e9
    print(f"   FLOPs (per image): {flops_g:.4f} GFLOPs")

    # 3. Data (FashionMNIST as per Q2)
    transform = transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    data_root = '/csehome/m25csa013/data'
    ds_train = datasets.FashionMNIST(root=data_root, train=True, download=True, transform=transform)
    ds_test = datasets.FashionMNIST(root=data_root, train=False, download=True, transform=transform)

    # Use Full Dataset
    train_loader = DataLoader(ds_train, batch_size=bs, shuffle=True, num_workers=4, pin_memory=(device_str=='cuda'))
    test_loader = DataLoader(ds_test, batch_size=bs, shuffle=False, num_workers=4, pin_memory=(device_str=='cuda'))

    # 4. Training (1 Epoch Loop)
    if opt_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    criterion = nn.CrossEntropyLoss()

    model.train()

    # Warmup
    if device_str == 'cuda':
        dummy = torch.randn(bs, 3, 64, 64).to(device)
        model(dummy).sum().backward()
        torch.cuda.synchronize()

    t0 = time.time()

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        if device_str == 'cuda':
            torch.cuda.synchronize()

    t1 = time.time()
    train_time_ms = (t1 - t0) * 1000


    # 5. Test Accuracy
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)

    accuracy = (correct / total) * 100
    print(f"   Test Accuracy: {accuracy:.2f}%")

    # Save Model (Global Best Only)
    global GLOBAL_BEST_ACC
    if accuracy > GLOBAL_BEST_ACC:
        GLOBAL_BEST_ACC = accuracy
        os.makedirs("models", exist_ok=True)
        save_path = "models/best_model_q2_gpu.pth"
        torch.save(model.state_dict(), save_path)

    return train_time_ms, flops_g, accuracy

# ------------------------------------------------------------------
# Main
# ------------------------------------------------------------------
if __name__ == "__main__":

    configs = [
        # GPU
        ('cuda', 16, 'SGD', 0.001),
        ('cuda', 16, 'Adam', 0.001),
    ]

    models_to_test = ["ResNet-18", "ResNet-50"]

    print("\n" + "="*120)
    print(f"{'Device':<6} | {'Model':<10} | {'BS':<3} | {'Opt':<5} | {'LR':<7} | {'Train Time (ms)':<16} | {'GFLOPs':<10} | {'Acc (%)':<8}")
    print("="*120)

    for device_str, bs, opt, lr in configs:
        if device_str == 'cuda' and not torch.cuda.is_available():
            print("CUDA not available. Exiting GPU benchmark.")
            break

        for model_name in models_to_test:
            try:
                t_ms, gflops, acc = run_benchmark(model_name, device_str, bs, opt, lr)
                print(f"{device_str:<6} | {model_name:<10} | {bs:<3} | {opt:<5} | {lr:<7} | {t_ms:<16.2f} | {gflops:<10.4f} | {acc:<8.2f}")
            except Exception as e:
                print(f"ERROR: {model_name} on {device_str} - {e}")
                import traceback
                traceback.print_exc()

```text
========================================================================================================================
Device | Model      | BS  | Opt   | LR      | Train Time (ms)  | GFLOPs     | Acc (%)
========================================================================================================================

>>> Benchmark: ResNet-18 on CUDA | BS=16 | Opt=SGD | LR=0.001
   FLOPs (per image): 0.2961 GFLOPs
   Test Accuracy: 88.41%
cuda   | ResNet-18  | 16  | SGD   | 0.001   | 20336.38         | 0.2961     | 88.41   

>>> Benchmark: ResNet-50 on CUDA | BS=16 | Opt=SGD | LR=0.001
   FLOPs (per image): 0.6673 GFLOPs
   Test Accuracy: 85.49%
cuda   | ResNet-50  | 16  | SGD   | 0.001   | 46177.53         | 0.6673     | 85.49   

>>> Benchmark: ResNet-18 on CUDA | BS=16 | Opt=Adam | LR=0.001
   FLOPs (per image): 0.2961 GFLOPs
   Test Accuracy: 88.26%
cuda   | ResNet-18  | 16  | Adam  | 0.001   | 25460.17         | 0.2961     | 88.26   

>>> Benchmark: ResNet-50 on CUDA | BS=16 | Opt=Adam | LR=0.001
   FLOPs (per image): 0.6673 GFLOPs
   Test Accuracy: 84.07%
cuda   | ResNet-50  | 16  | Adam  | 0.001   | 52566.23         | 0.6673     | 84.07
```